In [ ]:
from pathlib import Path
import sys
import matplotlib as mpl
CAMERA_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'figure_generation_notebooks' else Path.cwd().resolve()
EXPERIMENT_ROOT = Path('/mnt/e/watermark_rev2')
FIGURETYPE1_DIR = CAMERA_ROOT / 'figuretype1'
FIGURETYPE1_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EXPERIMENT_ROOT))
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
sys.path.insert(0, str(EXPERIMENT_ROOT))
from theorytools import KL_theory, deltaP
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


In [ ]:
results_dir = EXPERIMENT_ROOT / 'results_final'
result_files = sorted([f for f in results_dir.glob('GRID_*.json')])
for usemodel in ['opt-125m', 'pythia-160m', 'gpt2']:
    for usedataset in ['c4', 'lfqa', 'wikipedia']:
        include = [f for f in result_files if usemodel in f.name and usedataset in f.name]
        full_data_dict = {}
        for file_path in include:
            with open(file_path, 'r') as f:
                data = json.load(f)
            seed = data['config']['seed']
            temp = data['config']['temperature']
            gamma = data['config']['gamma']
            delta = data['config']['delta']
            if seed > 0:
                continue
            if gamma not in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
                continue
            if delta not in [0.5, 1, 2]:
                continue
            full_data_dict[temp, gamma, delta, seed] = {'gt': [], 'kl': []}
            for result in data['results']:
                ngram = result['watermarked']['ngram_info']
                gt = np.array(result['watermarked']['green_token_mask'])
                kl = np.array(result['watermarked']['kl_divergence'][1:])
                ng = np.array(ngram)
                if len(gt) != len(ng) or len(gt) != len(kl):
                    continue
                gt = gt[ng == True]
                kl = kl[ng == True]
                full_data_dict[temp, gamma, delta, seed]['gt'].extend(gt)
                full_data_dict[temp, gamma, delta, seed]['kl'].extend(kl)
        if not full_data_dict:
            continue
        df = []
        for k, v in full_data_dict.items():
            temp, gamma, delta, seed = k
            theoretical_KL = KL_theory(gamma, delta * temp, temp=temp)
            theoretical_deltaP = deltaP(gamma, delta * temp, temp=temp)
            df.append({'Temperature': temp, 'Gamma': gamma, 'Delta': delta, 'Theoretical GTR': theoretical_deltaP + gamma, 'Empirical GTR': np.average(v['gt'])})
        df = pd.DataFrame(df)
        x = df['Theoretical GTR'].values
        y = df['Empirical GTR'].values
        k = np.sum(x * y) / np.sum(x * x)
        plt.figure(figsize=(8, 5), dpi=150)
        plt.scatter(x, y, alpha=0.7)
        x_line = np.linspace(x.min(), x.max(), 100)
        plt.plot(x_line, k * x_line, 'r-', label=f'y = {k:.3f}x', linewidth=2)
        plt.xlabel('Theoretical Green Token Rate')
        plt.ylabel('Empirical Green Token Rate')
        plt.title(f'GRID: {usemodel} / {usedataset}')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.savefig(FIGURETYPE1_DIR / f'green_token_rate_GRID_{usemodel}_{usedataset}.pdf', dpi=300, bbox_inches='tight')
        plt.show()


In [ ]:
results_dir = EXPERIMENT_ROOT / 'results_final'
result_files = sorted([f for f in results_dir.glob('GRID_*.json')])
r2_results = []
for usemodel in ['opt-125m', 'pythia-160m', 'gpt2']:
    for usedataset in ['c4', 'lfqa', 'wikipedia']:
        include = [f for f in result_files if usemodel in f.name and usedataset in f.name]
        full_data_dict = {}
        for file_path in include:
            with open(file_path, 'r') as f:
                data = json.load(f)
            seed, temp = (data['config']['seed'], data['config']['temperature'])
            gamma, delta = (data['config']['gamma'], data['config']['delta'])
            if seed > 0 or gamma not in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9] or delta not in [0.5, 1, 2]:
                continue
            full_data_dict[temp, gamma, delta, seed] = {'gt': [], 'kl': []}
            for result in data['results']:
                ngram = result['watermarked']['ngram_info']
                gt = np.array(result['watermarked']['green_token_mask'])
                kl = np.array(result['watermarked']['kl_divergence'][1:])
                ng = np.array(ngram)
                if len(gt) != len(ng) or len(gt) != len(kl):
                    continue
                full_data_dict[temp, gamma, delta, seed]['gt'].extend(gt[ng == True])
                full_data_dict[temp, gamma, delta, seed]['kl'].extend(kl[ng == True])
        df_list = []
        for k, v in full_data_dict.items():
            temp, gamma, delta, seed = k
            df_list.append({'Theoretical GTR': deltaP(gamma, delta * temp, temp=temp) + gamma, 'Empirical GTR': np.average(v['gt']), 'Theoretical KL': KL_theory(gamma, delta * temp, temp=temp), 'Empirical KL': np.average(v['kl'])})
        if not df_list:
            continue
        df = pd.DataFrame(df_list)
        x, y = (df['Theoretical GTR'].values, df['Empirical GTR'].values)
        k = np.sum(x * y) / np.sum(x * x)
        r2_gtr = 1 - np.sum((y - k * x) ** 2) / np.sum((y - np.mean(y)) ** 2)
        x, y = (df['Theoretical KL'].values, df['Empirical KL'].values)
        k = np.sum(x * y) / np.sum(x * x)
        r2_kl = 1 - np.sum((y - k * x) ** 2) / np.sum((y - np.mean(y)) ** 2)
        r2_results.append({'Model': usemodel, 'Dataset': usedataset, 'GTR R²': round(r2_gtr, 4), 'KL R²': round(r2_kl, 4)})
df_r2 = pd.DataFrame(r2_results)


In [ ]:
import seaborn as sns
FONTSIZE_TITLE = 25
FONTSIZE_AXLABEL = 25
FONTSIZE_LEGEND = 18
FONTSIZE_TICK = 18
FONTSIZE_SUBTITLE = 20
FONTSIZE_MODEL = 20
TICK_NBINS = 4
ASPECT_EQUAL = True
SCATTER_SIZE = 50
SCATTER_ALPHA = 0.9
SCATTER_ZORDER = 5
LINE_FIT_WIDTH = 4
LINE_FIT_ALPHA = 0.8
LINE_FIT_ZORDER = 4
LINE_REF_WIDTH = 2
LINE_REF_ALPHA = 0.8
LINE_REF_ZORDER = 8
PALETTE = 'deep'
COLOR_SCATTER = sns.color_palette('muted')[0]
COLOR_FIT = sns.color_palette('muted')[2]
COLOR_REF = 'gray'
sns.set_theme(style='white', context='talk', font_scale=0.8)
results_dir = EXPERIMENT_ROOT / 'results_final'
result_files = sorted([f for f in results_dir.glob('GRID_*.json')])
models = ['opt-125m', 'pythia-160m', 'gpt2']
datasets = ['c4', 'lfqa', 'wikipedia']
fig, axes = plt.subplots(3, 3, figsize=(12, 12), sharex=True, sharey=True)
for row, usemodel in enumerate(models):
    for col, usedataset in enumerate(datasets):
        ax = axes[row, col]
        include = [f for f in result_files if usemodel in f.name and usedataset in f.name]
        full_data_dict = {}
        for file_path in include:
            with open(file_path, 'r') as f:
                data = json.load(f)
            seed, temp = (data['config']['seed'], data['config']['temperature'])
            gamma, delta = (data['config']['gamma'], data['config']['delta'])
            if seed > 0 or gamma not in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8] or delta not in [0.5, 1, 2]:
                continue
            full_data_dict[temp, gamma, delta, seed] = {'gt': []}
            for result in data['results']:
                ngram = result['watermarked']['ngram_info']
                gt = np.array(result['watermarked']['green_token_mask'])
                ng = np.array(ngram)
                if len(gt) != len(ng):
                    continue
                full_data_dict[temp, gamma, delta, seed]['gt'].extend(gt[ng == True])
        df_list = []
        for k, v in full_data_dict.items():
            temp, gamma, delta, seed = k
            df_list.append({'Theoretical GTR': deltaP(gamma, delta * temp, temp=temp) + gamma, 'Empirical GTR': np.average(v['gt'])})
        df = pd.DataFrame(df_list)
        x = df['Theoretical GTR'].values
        y = df['Empirical GTR'].values
        k = np.sum(x * y) / np.sum(x * x)
        y_pred = k * x
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = 1 - ss_res / ss_tot
        lims = [0.2, 1.0]
        ax.plot(lims, lims, linestyle='--', linewidth=LINE_REF_WIDTH, color=COLOR_REF, alpha=LINE_REF_ALPHA, zorder=LINE_REF_ZORDER)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, k * x_line, linewidth=LINE_FIT_WIDTH, alpha=LINE_FIT_ALPHA, color=COLOR_FIT, label='Fitted', zorder=LINE_FIT_ZORDER)
        ax.scatter(x, y, s=SCATTER_SIZE, alpha=SCATTER_ALPHA, color=COLOR_SCATTER, label='Sample', edgecolor='none', zorder=SCATTER_ZORDER)
        if ASPECT_EQUAL:
            ax.set_aspect('equal', adjustable='box')
            ax.set_xlim(0.2, 1.0)
            ax.set_ylim(0.2, 1.0)
        if row == 0:
            ax.set_title(f'{usedataset.upper()}', fontsize=FONTSIZE_SUBTITLE)
        if col == 2:
            ax.annotate(usemodel.upper(), xy=(1.05, 0.5), xycoords='axes fraction', fontsize=FONTSIZE_MODEL, va='center', rotation=-90)
        ax.legend(frameon=False, fontsize=FONTSIZE_LEGEND, loc='lower right')
        ax.tick_params(labelsize=FONTSIZE_TICK)
        ax.locator_params(axis='both', nbins=TICK_NBINS)
        ax.grid(False)
        sns.despine(ax=ax)
fig.supxlabel('Theoretical Green Token Rate', fontsize=FONTSIZE_AXLABEL)
fig.supylabel('$H_1$ Empirical Green Token Rate', fontsize=FONTSIZE_AXLABEL)
plt.tight_layout()
plt.savefig(FIGURETYPE1_DIR / 'green_token_rate_GRID_all_combined.pdf', dpi=300, bbox_inches='tight')
plt.show()
